In [ ]:
from pydantic import BaseModel, Field, ConfigDict
from json_dummy import dummyJSON
import numpy as np

<class 'str'>


In [ ]:
#Create the structure basemodel for validation
class Node(BaseModel):
    id: int
    x: float
    y: float


class Member(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    id: int
    node1: int
    node2: int
    area: float
    I: float
    E: float
    length: float |None=None
    angle: float |None=None
    connectivity: list|None=None
    trussqn: bool | None=False
    udl: float | None = 0.0


class BoundaryCondition(BaseModel):
    id: int
    bc_type: str = Field(alias="BC type")
    node: int


class PointLoad(BaseModel):
    id: int
    node: int
    Fx: float
    Fy: float


class Moment(BaseModel):
    id: int
    node: int
    M: float


class UDL(BaseModel):
    id: int
    member: int
    w: float


class Structure(BaseModel):
    nodes: list[Node]
    members: list[Member]
    boundaryConditions: list[BoundaryCondition]
    pointLoads: list[PointLoad]
    moments: list[Moment]
    udls: list[UDL]
    trussMemberIds: list[int]
    frameMembers: list[Member]
    trussNodes: list[int]
    frameNodes: list[Node]

In [3]:
#extract the structure from json string
structure = Structure.model_validate_json(dummyJSON) #converts the json string to a structure

In [ ]:
from typing import NamedTuple

class NodeInfo(NamedTuple):
    index: int 
    node: Node  

node_info = {n.id: NodeInfo(i, n) for i, n in enumerate(structure.nodes)}
dofs_per_node = 3

def dof_base(node_id: int) -> int:
    """Base DOF index (ux) for a given node id. Raises if id is unknown."""
    try:
        return node_info[node_id].index * dofs_per_node
    except KeyError:
        raise ValueError(f"Reference to unknown node id {node_id}")


In [6]:
#element connectivity
n_elems = len(structure.members)
n_nodes = len(structure.nodes)
ele_connectivity = np.zeros((n_elems,2))
for i,member in enumerate(structure.members):
    ele_connectivity[i][0] = member.node1
    ele_connectivity[i][1] = member.node2
    b1 = dof_base(member.node1)
    b2 = dof_base(member.node2)
    member.connectivity = [b1, b1+1, b1+2, b2, b2+1, b2+2]
    for tid in structure.trussMemberIds:
        if tid == member.id:
            member.trussqn = True 
    for udl in structure.udls:
        if udl.member == member.id:
            member.udl = udl.w

In [7]:
from collections import defaultdict

connected_types = defaultdict(list)

for member in structure.members:
    connected_types[member.node1].append(member.trussqn)
    connected_types[member.node2].append(member.trussqn)

truss_nodes = {
    node_id
    for node_id, is_truss in connected_types.items()
    if all(is_truss)  # every connected element is a truss
}

In [8]:
total_dofs = n_nodes * dofs_per_node

active_dofs = []

for node in structure.nodes:
    base = dof_base(node.id)

    # Always keep ux and uy
    active_dofs.extend([base, base + 1])

    # Keep theta only for frame nodes
    if node.id not in truss_nodes:
        active_dofs.append(base + 2)

active_dofs = np.array(active_dofs)

In [9]:
active_dofs

array([0, 1, 2, 3, 4, 5, 6, 7])

In [10]:
#Connectivity matrix
dofs_per_node = 3
nodes_per_element = 2
dofs_per_element = dofs_per_node*nodes_per_element
connectivity_matrix = np.zeros((n_elems, dofs_per_element))
for element in range(n_elems):
    for i, node in enumerate(ele_connectivity[element]):
        connectivity_matrix[element,
                            i*dofs_per_node:i*dofs_per_node+3]= [(element*dofs_per_element+i*dofs_per_node), 
                                                                (element*dofs_per_element+i*dofs_per_node+1), 
                                                                (element*dofs_per_element+i*dofs_per_node+2)]

In [ ]:
# Sanity check
for member in structure.members:
    b1 = dof_base(member.node1)
    b2 = dof_base(member.node2)
    expected = [b1, b1+1, b1+2, b2, b2+1, b2+2]
    assert member.connectivity == expected
    print(member.connectivity)

[0, 1, 2, 3, 4, 5]
[6, 7, 8, 3, 4, 5]


In [12]:
element_lengths = np.empty(len(structure.members))
element_orientation = np.empty(len(structure.members))

for i, member in enumerate(structure.members):
    n1 = node_info[member.node1].node
    n2 = node_info[member.node2].node

    dx = n2.x - n1.x
    dy = n2.y - n1.y

    member.length = np.hypot(dx, dy)
    member.angle = np.arctan2(dy, dx)

In [13]:
def axial_sf(xi):
    """Lagrange linear shape functions [N1, N2] at natural coord xi."""
    return np.array([0.5*(1.0 - xi), 0.5*(1.0 + xi)])

def axial_dsf(xi, he):
    """First derivative dN/dxbar = dN/dxi * (2/he)."""
    return np.array([-0.5, 0.5]) * (2.0 / he)
  
def hermite_sf(xi, he):
    """Hermite cubic shape functions [phi1, phi2, phi3, phi4] at xi."""
    return np.array([
         0.25*(2.0 - 3.0*xi + xi**3),
        (he/8.0)*(1.0 - xi - xi**2 + xi**3),
         0.25*(2.0 + 3.0*xi - xi**3),
        (he/8.0)*(-1.0 - xi + xi**2 + xi**3),
    ])


def hermite_ddsf(xi, he):
    """Second derivative d2phi/dxbar2 = (2/he)^2 * d2phi/dxi2 at xi."""
    d2phi_dxi2 = np.array([
         0.25  * 6.0*xi,
        (he/8.0)*(-2.0 + 6.0*xi),
         0.25  *(-6.0*xi),
        (he/8.0)*( 2.0 + 6.0*xi),
    ])
    return d2phi_dxi2 * (2.0/he)**2

def hermite_dddsf(xi, he):
    """Third derivative for finding the shear force in terms of transverse disp"""
    d3phi_dxi3 = np.array([
         0.25  * 6.0,
        (he/8.0)* 6.0,
         0.25  *(-6.0),
        (he/8.0)* 6.0,
    ])
    return d3phi_dxi3 * (2.0/he)**3



In [14]:
def transformation_matrix(angle):
    """6x6 local-to-global transformation matrix for a 2D frame element."""
    c, s = np.cos(angle), np.sin(angle)
    return np.array([
        [ c,  s,  0,  0,  0,  0],
        [-s,  c,  0,  0,  0,  0],
        [ 0,  0,  1,  0,  0,  0],
        [ 0,  0,  0,  c,  s,  0],
        [ 0,  0,  0, -s,  c,  0],
        [ 0,  0,  0,  0,  0,  1],
    ])


In [15]:
def transformation_matrix2(angle):
    """4x4 local-to-global transformation matrix for a 2D truss element."""
    c, s = np.cos(angle), np.sin(angle)
    return np.array([
        [ c,  s,  0,  0],
        [-s,  c,  0,  0],
        [ 0,  0,  c,  s],
        [ 0,  0, -s,  c]
    ])


In [16]:
total_dofs = n_nodes*dofs_per_node

KG = np.zeros((total_dofs, total_dofs))
FG = np.zeros(total_dofs)

g_pts = 4
xi_gp, w_gp = np.polynomial.legendre.leggauss(g_pts)

for i,member in enumerate(structure.members):
    he = member.length
    Je = 0.5*he
    EA = (member.E)*(member.area)
    EI = (member.E)*(member.I)
    conn_el = member.connectivity
    angle = member.angle
    udl = member.udl

    ke_local = np.zeros((dofs_per_element, dofs_per_element))
    fe_local = np.zeros(dofs_per_element)

    if member.trussqn == False:
        for gpt in range(g_pts):
            xi = xi_gp[gpt]
            w = w_gp[gpt]

            dNa = axial_dsf(xi, he)
            Ba = np.zeros(dofs_per_element)
            Ba[0] = dNa[0]; Ba[3] = dNa[1]

            ddPhi = hermite_ddsf(xi, he)
            Bb = np.zeros(dofs_per_element)
            Bb[1] = ddPhi[0]; Bb[2] = ddPhi[1]
            Bb[4] = ddPhi[2]; Bb[5] = ddPhi[3]
            
            ke_local += (EA * np.outer(Ba, Ba) + EI * np.outer(Bb, Bb)) * w *Je
            Phi  = hermite_sf(xi, he)
            fe_b = np.zeros(dofs_per_element)
            fe_b[1] = Phi[0];  fe_b[2] = Phi[1]
            fe_b[4] = Phi[2];  fe_b[5] = Phi[3]
            fe_local += member.udl * fe_b * w * Je   
    else:
        for gpt in range(g_pts):
            xi = xi_gp[gpt]
            w = w_gp[gpt]
            dNa = axial_dsf(xi, he)
            Ba = np.zeros(dofs_per_element)
            Ba[0] = dNa[0]; Ba[3] = dNa[1]
                         
            ke_local += EA * np.outer(Ba, Ba) * w *Je
               

    # Transform local -> global:  K_global = T^T k_local T
    T = transformation_matrix(angle)
    ke_global = T.T @ ke_local @ T
    fe_global = T.T @ fe_local
    # Assemble into global K and F
    for nr in range(dofs_per_element):
        r = conn_el[nr]
        FG[r] += fe_global[nr]
        for nc in range(dofs_per_element):
            c = conn_el[nc]
            KG[r, c] += ke_global[nr, nc]

In [17]:
#Add the pointload
for pointload in structure.pointLoads:
    base = dof_base(pointload.node)
    FG[base] += pointload.Fx
    FG[base + 1] += pointload.Fy

#Add the moment
for moment in structure.moments:
    base = dof_base(moment.node)
    FG[base + 2] += moment.M

In [ ]:
KG_copy = KG.copy()
FG_copy = FG.copy()

In [19]:
#Applying the boundary conditions
for BC in structure.boundaryConditions:
    nd1 = dof_base(BC.node)
    nd2 = nd1 + 1
    nd3 = nd1 + 2

    if BC.bc_type == "fixed":
        constrained = (nd1, nd2, nd3)
    elif BC.bc_type == "pinned":
        constrained = (nd1, nd2)
    elif BC.bc_type == "roller":
        constrained = (nd2,)
    else:
        constrained = ()

    for d in constrained:
        KG[d, d] = 1
        FG[d] = 0

    for i in range(total_dofs):
        for d in constrained:
            if i != d:
                KG[d, i] = 0.0
                KG[i, d] = 0.0

In [20]:
KG = KG[np.ix_(active_dofs, active_dofs)]
FG = FG[active_dofs]

In [22]:
U_final = np.linalg.solve(KG, FG)

In [23]:
U_final_global = np.zeros(total_dofs)
U_final_global[active_dofs] = U_final
R_final_global = np.zeros(total_dofs)
R_final_global=KG_copy@U_final_global-FG_copy

In [ ]:
class NodalResult(BaseModel):
    id: int
    u: float 
    v: float
    theta: float | None=None
    R_x: float
    R_y: float
    R_M: float | None=None

class MemberResult(BaseModel):
    id: int
    u: list[float]
    v: list[float]
    x_def_coord: list[float]
    y_def_coord: list[float]
    axial_stress: list[float]
    axial_strain: list[float]
    axial_force: list[float] 
    shear_force: list[float] |None = None
    bending_moment: list[float] |None = None

class PostProcess(BaseModel):
    nodal_results: list[NodalResult]
    member_results: list[MemberResult]

In [25]:
postprocess = PostProcess(
    nodal_results=[],
    member_results=[]
)

In [26]:
for node in structure.nodes:

    dof_u = dof_base(node.id)
    dof_v = dof_u + 1
    dof_theta = dof_u + 2

    postprocess.nodal_results.append(
        NodalResult(
            id=node.id,
            u=U_final_global[dof_u],
            v=U_final_global[dof_v],
            theta=U_final_global[dof_theta],
            R_x=R_final_global[dof_u],
            R_y=R_final_global[dof_v],
            R_M=R_final_global[dof_theta]
        )
    )

In [27]:
#Postprcessing the results
n_plot = 80
xi_plot = np.linspace(-1,1,n_plot)
scale = 100


for member in structure.members:
    x_def = []
    y_def = []
    AxialStress= []
    AxialStrain = []
    AxialForce = []
    BendingMoment = []
    ShearForce = []
    n1 = member.node1
    n2 = member.node2
    angle = member.angle
    c, s = np.cos(angle), np.sin(angle)
    he = member.length
    s_along = 0.5*he*(xi_plot+1)
    conn = member.connectivity
    T = transformation_matrix(angle)
    conn = np.asarray(conn)
    Ue_local = T@U_final_global[conn]
    dNa = axial_dsf(0, he)
    E = member.E
    area = member.area
    x1 = np.ones(n_plot)*node_info[n1].node.x
    y1 = np.ones(n_plot)*node_info[n1].node.y
    x_initial = x1 + s_along*c
    y_initial = y1 + s_along*s

    if member.trussqn==False:
        EI = E*member.I

        for k, xi in enumerate(xi_plot):
            Na = axial_sf(xi)
            u_bar = Na[0]*Ue_local[0] + Na[1]*Ue_local[3]
            du_bar = dNa@np.array([Ue_local[0], Ue_local[3]])
            axial_stress = E*du_bar
            axial_force = axial_stress*area
            Phi = hermite_sf(xi, he)
            v_bar = (Phi[0]*Ue_local[1] + Phi[1]*Ue_local[2]
               + Phi[2]*Ue_local[4] + Phi[3]*Ue_local[5])
            d2Phi = hermite_ddsf(xi, he)
            d2v_bar = d2Phi@np.array([Ue_local[1], Ue_local[2], Ue_local[4], Ue_local[5]])
            BM = -EI*d2v_bar
            d3Phi = hermite_dddsf(xi, he)
            d3v_bar = d3Phi@np.array([Ue_local[1], Ue_local[2], Ue_local[4], Ue_local[5]])
            SF = -EI*d3v_bar
          
            R2          = np.array([[c, -s], [s, c]])
            disp_global = R2 @ np.array([u_bar, v_bar])
            x_def.append(disp_global[0])
            y_def.append(disp_global[1])
            AxialStress.append(axial_stress)
            BendingMoment.append(BM)
            AxialForce.append(axial_force)
            ShearForce.append(SF)
            AxialStrain.append(du_bar)

        x_final = x_initial + x_def
        y_final = y_initial + y_def

        postprocess.member_results.append(MemberResult(
        id=member.id,
        u= x_def,
        v= y_def,
        x_def_coord= x_final,
        y_def_coord= y_final,
        axial_force= AxialForce,
        axial_strain= AxialStrain,
        axial_stress= AxialStress,
        bending_moment= BendingMoment,
        shear_force= ShearForce            
    ))   

    if member.trussqn==True:

        U1x, U1y = U_final_global[conn[0]], U_final_global[conn[1]]
        U2x, U2y = U_final_global[conn[3]], U_final_global[conn[4]]

        for k, xi in enumerate(xi_plot):
            Na = axial_sf(xi)
            u_bar = Na[0]*Ue_local[0] + Na[1]*Ue_local[3]
            du_bar = dNa@np.array([Ue_local[0], Ue_local[3]])
            axial_stress = E*du_bar
            axial_force = axial_stress*area
            u_g = Na[0]*U1x + Na[1]*U2x 
            v_g = Na[0]*U1y + Na[1]*U2y
            x_def.append(u_g)
            y_def.append(v_g)
            AxialStress.append(axial_stress)
            AxialForce.append(axial_force)
            AxialStrain.append(du_bar)

            
        x_final = x_initial + x_def
        y_final = y_initial + y_def
        
        postprocess.member_results.append(MemberResult(
            id=member.id,
            u= x_def,
            v= y_def,
            x_def_coord= x_final,
            y_def_coord= y_final,
            axial_force= AxialForce,
            axial_strain= AxialStrain,
            axial_stress= AxialStress,             
        ))        
